In [132]:
!pip install -q sentence-transformers faiss-cpu transformers datasets rank_bm25 rouge-score torch

Importar data

In [133]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:200]")

# Filtramos artículos muy cortos
texts = [x["article"] for x in dataset if len(x["article"]) > 100]

print("Documents loaded:", len(texts))
print("Example text (first 500 chars):\n", texts[0][:500])

Documents loaded: 200
Example text (first 500 chars):
 LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s


Chunk

In [134]:
def chunk_text(text, size=700, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        chunk = text[start:start + size]
        if len(chunk) > 150:
            chunks.append(chunk)
        start += size - overlap
    return chunks

chunks = []
for idx, article in enumerate(texts):
    article_chunks = chunk_text(article)
    chunks.extend(article_chunks)

print(f"Total chunks: {len(chunks)}")
print("Ejemplo chunk:", chunks[0][:500])
print("chunks[:10] OK:", len(chunks[:10]))
chunks[0:2]  # Para ver en output


Total chunks: 1474
Ejemplo chunk: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s
chunks[:10] OK: 10


['LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagan',
 'oon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and 

In [135]:
chunks = []
for idx, article in enumerate(texts):
    article_chunks = chunk_text(article, size=700, overlap=200)
    for c in article_chunks:
        chunks.append({'text': c, 'article_id': idx})

print(f"Total chunks: {len(chunks)}")
print("Ejemplo chunk:", chunks[0]['text'][:500])
print("chunks[:10] OK:", len(chunks[:10]))

Total chunks: 1474
Ejemplo chunk: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s
chunks[:10] OK: 10


In [136]:
chunks[:10]

[{'text': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagan',
  'article_id': 0},
 {'text': 'oon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost

In [137]:
current_article_id = None

def identify_article(question):
    global current_article_id
    docs = hybrid_search(question, 10)
    current_article_id = docs[0]['article_id']
    print(f"🔍 Artículo detectado: ID={current_article_id}")
    return current_article_id

def rag_article_aware(question):
    global current_article_id

    if current_article_id is None:
        current_article_id = identify_article(question)

    relevant_docs = [doc for doc in chunks if doc['article_id'] == current_article_id]
    docs_text = [doc['text'] for doc in relevant_docs]

    reranked_docs = rerank(question, docs_text, RERANK_OPTIMO)
    context = "\n".join(reranked_docs)
    answer = ask_llm(context, question)
    grounding_score = grounding(answer, reranked_docs)

    return answer, grounding_score

def reset_article_session():
    global current_article_id
    current_article_id = None
    print("🔄 Sesión reseteada")


In [138]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embedder = SentenceTransformer('multi-qa-mpnet-base-dot-v1')

embeddings = embedder.encode(chunks, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexHNSWFlat(dim, 32)
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 100
index.add(embeddings)

print(f"FAISS index size: {index.ntotal}")
print(f"Embedding shape: {embeddings.shape}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

FAISS index size: 1474
Embedding shape: (1474, 768)


In [140]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [c["text"].split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

from rank_bm25 import BM25Okapi
import faiss
import numpy as np

def hybrid_search(query, k=5):
    # BM25 global
    bm25_scores = bm25.get_scores(query.split())
    # FAISS global
    qemb = embedder.encode([query]).astype('float32')
    faiss.normalize_L2(qemb)
    D, I = index.search(qemb, 20)
    # Fusion alpha=0.7 para dense
    combined = 0.7 * (-D[0]) + 0.3 * bm25_scores[I[0]]
    top_k = np.argsort(combined)[-k:]
    return [chunks[i] for i in top_k]


In [141]:
from sentence_transformers import CrossEncoder

try:
    reranker
except:
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, docs, topk=3):
    """Compatible con STRINGS únicamente"""
    pairs = [[query, doc] for doc in docs]  # docs son strings
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:topk]]

In [142]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# FIX: Configurar pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

def ask_llm_agresivo(context, question):
    """Fuerza extracción INCLUSIVA"""
    prompt = f"""Contexto:
{context}

Pregunta: {question}

Instrucciones:
1. Busca CUALQUIER mención relacionada
2. Extrae NÚMEROS, NOMBRES, FRASES exactas
3. Si hay info parcial → úsala
4. Solo "Not found" si NO HAY NADA relacionado

Respuesta:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512, padding=True)

    with torch.no_grad():  # ← no_inference_mode para T5
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,  # ← FIX warning
            max_new_tokens=max_length,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
            num_beams=4,  # ← Beam search más preciso
            early_stopping=True
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Limpiar prompt de respuesta
    return answer.split("Answer:")[-1].strip()


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

In [143]:
def add_citations(answer, docs):
    """
    Agrega referencias [docX] solo si alguna palabra clave de la oración aparece en el chunk correspondiente.
    """
    sentences = answer.split(".")
    cited = []

    for s in sentences:
        s_clean = s.strip()
        if len(s_clean) == 0:
            continue

        found = False
        for i, doc in enumerate(docs):
            for w in s_clean.split():
                if w.lower() in doc["text"].lower():
                    cited.append(f"{s_clean} [doc{i}]")
                    found = True
                    break
            if found:
                break
        if not found:
            cited.append(s_clean)
    return ". ".join(cited)

In [144]:
def hallucination_guard(answer, context):
    hits = sum(1 for w in answer.split() if w.lower() in context.lower())
    return hits / max(len(answer.split()),1)

def grounding(answer, retrieved_chunks):
    ctx = " ".join(retrieved_chunks)
    hits = sum(1 for w in answer.split() if w.lower() in ctx.lower())
    return hits / max(len(answer.split()), 1)

In [145]:
from rouge_score import rouge_scorer

def evaluate_answer(reference, prediction):
    scorer = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)
    scores = scorer.score(reference, prediction)
    return scores

In [146]:
questions = [
    "How much money did Daniel Radcliffe inherit when he turned 18?",
    "How does he plan to spend his fortune?",
    "What does he like to buy?",
    "Has he earned money from the Harry Potter films?",
    "What other projects has he done?"
]

print("=== RAG PRODUCTION TEST ===\n")

for i, question in enumerate(questions, 1):
    # Pipeline COMPLETO y limpio
    docs = hybrid_search(question, 5)
    docs_text = [doc['text'] for doc in docs]
    reranked_docs = rerank(question, docs_text, 3)
    context = "\n".join(reranked_docs)

    answer = ask_llm(context, question)

    print(f"{i}. Q: {question}")
    print(f"   A: {answer}\n")

=== RAG PRODUCTION TEST ===

1. Q: How much money did Daniel Radcliffe inherit when he turned 18?
   A: £20 million ($41.1 million

2. Q: How does he plan to spend his fortune?
   A: Not found in context

3. Q: What does he like to buy?
   A: things that cost about 10 pounds

4. Q: Has he earned money from the Harry Potter films?
   A: Not found in context

5. Q: What other projects has he done?
   A: He will also appear in "December Boys



In [147]:
questions = [
    "How much money did Daniel Radcliffe inherit when he turned 18?",
    "How does he plan to spend his fortune?",
    "What does he like to buy?",
    "Has he earned money from the Harry Potter films?",
    "What other projects has he done?"
]

print("=== RAG CON GROUNDING SCORES ===\n")
scores = []

for i, question in enumerate(questions, 1):
    docs = hybrid_search(question, 10)
    docs_text = [doc['text'] for doc in docs]
    reranked_docs = rerank(question, docs_text, 5)

    answer = ask_llm("\n".join(reranked_docs), question)

    # 🔥 GROUNDING CORREGIDO
    grounding_score = grounding(answer, reranked_docs)

    print(f"{i}. Q: {question}")
    print(f"   A: {answer}")
    print(f"   🟢 GROUNDING: {grounding_score:.3f}\n")
    scores.append(grounding_score)

print(f"📊 PROMEDIO: {np.mean(scores):.3f}")

=== RAG CON GROUNDING SCORES ===

1. Q: How much money did Daniel Radcliffe inherit when he turned 18?
   A: £20 million ($41.1 million)
   🟢 GROUNDING: 1.000

2. Q: How does he plan to spend his fortune?
   A: Not found in context
   🟢 GROUNDING: 0.500

3. Q: What does he like to buy?
   A: books and CDs and DVDs
   🟢 GROUNDING: 1.000

4. Q: Has he earned money from the Harry Potter films?
   A: Not found in context
   🟢 GROUNDING: 0.500

5. Q: What other projects has he done?
   A: Not found in context
   🟢 GROUNDING: 0.500

📊 PROMEDIO: 0.700


In [148]:
questions = [
    "How much money did Daniel Radcliffe inherit when he turned 18?",
    "How does he plan to spend his fortune?",
    "What does he like to buy?",
    "Has he earned money from the Harry Potter films?",
    "What other projects has he done?"
]

results = []

# Grilla de parámetros
for k in [5, 8, 10, 12, 15, 20]:
    for topk_rerank in [3, 4, 5]:
        print(f"\n🔍 Testing k={k}, rerank={topk_rerank}")

        scores = []
        useful_answers = 0

        for question in questions:
            # Pipeline fijo
            docs = hybrid_search(question, k)
            docs_text = [doc['text'] for doc in docs]
            reranked_docs = rerank(question, docs_text, topk_rerank)

            answer = ask_llm("\n".join(reranked_docs), question)
            score = grounding(answer, reranked_docs)

            scores.append(score)
            if score > 0.3: useful_answers += 1

        avg_score = np.mean(scores)
        std_score = np.std(scores)
        useful_pct = useful_answers / len(questions)

        results.append({
            'k': k,
            'rerank': topk_rerank,
            'avg_grounding': avg_score,
            'std': std_score,
            'useful_pct': useful_pct,
            'best': avg_score
        })

        print(f"   📊 Avg: {avg_score:.3f}, Std: {std_score:.3f}, Útil: {useful_pct:.0%}")

# MEJOR combinación
best = max(results, key=lambda x: x['best'])
print(f"\n🎯 MEJOR: k={best['k']}, rerank={best['rerank']}")
print(f"   📈 Grounding: {best['avg_grounding']:.3f}")
print(f"   ✅ Útil: {best['useful_pct']:.0%}")



🔍 Testing k=5, rerank=3
   📊 Avg: 0.800, Std: 0.245, Útil: 100%

🔍 Testing k=5, rerank=4
   📊 Avg: 0.800, Std: 0.245, Útil: 100%

🔍 Testing k=5, rerank=5
   📊 Avg: 0.800, Std: 0.245, Útil: 100%

🔍 Testing k=8, rerank=3
   📊 Avg: 0.800, Std: 0.245, Útil: 100%

🔍 Testing k=8, rerank=4
   📊 Avg: 0.800, Std: 0.245, Útil: 100%

🔍 Testing k=8, rerank=5
   📊 Avg: 0.800, Std: 0.245, Útil: 100%

🔍 Testing k=10, rerank=3
   📊 Avg: 0.700, Std: 0.245, Útil: 100%

🔍 Testing k=10, rerank=4
   📊 Avg: 0.700, Std: 0.245, Útil: 100%

🔍 Testing k=10, rerank=5
   📊 Avg: 0.700, Std: 0.245, Útil: 100%

🔍 Testing k=12, rerank=3
   📊 Avg: 0.700, Std: 0.245, Útil: 100%

🔍 Testing k=12, rerank=4
   📊 Avg: 0.700, Std: 0.245, Útil: 100%

🔍 Testing k=12, rerank=5
   📊 Avg: 0.700, Std: 0.245, Útil: 100%

🔍 Testing k=15, rerank=3
   📊 Avg: 0.800, Std: 0.245, Útil: 100%

🔍 Testing k=15, rerank=4
   📊 Avg: 0.800, Std: 0.245, Útil: 100%

🔍 Testing k=15, rerank=5
   📊 Avg: 0.800, Std: 0.245, Útil: 100%

🔍 Testing k=20,

In [149]:
K_OPTIMO = 5
RERANK_OPTIMO = 3

def rag_pipeline_optimizado(question):
    """RAG con parámetros ÓPTIMOS fijos"""
    docs = hybrid_search(question, K_OPTIMO)
    docs_text = [doc['text'] for doc in docs]
    reranked_docs = rerank(question, docs_text, RERANK_OPTIMO)
    context = "\n".join(reranked_docs)
    answer = ask_llm(context, question)
    grounding_score = grounding(answer, reranked_docs)
    return answer, grounding_score


In [150]:
!pip install bert-score

def bertscore_semantico(answer, reference_chunks):
    from bert_score import score

    reference = " ".join(reference_chunks)
    P, R, F1 = score([answer], [reference], lang="en")
    return F1.item()

In [151]:
import pandas as pd
from bert_score import score
import warnings
warnings.filterwarnings("ignore")

questions = [
    "How much money did Daniel Radcliffe inherit when he turned 18?",
    "How does he plan to spend his fortune?",
    "What does he like to buy?",
    "Has he earned money from the Harry Potter films?",
    "What other projects has he done?"
]

print("=== DASHBOARD MÉTRICAS COMPLETO (BERTScore) ===\n")
metrics = []

for i, question in enumerate(questions, 1):
    answer, grounding_score = rag_pipeline_optimizado(question)
    docs = hybrid_search(question, K_OPTIMO)
    docs_text = [doc['text'] for doc in docs]
    reranked_docs = rerank(question, docs_text, RERANK_OPTIMO)

    reference = " ".join(reranked_docs)
    P, R, F1 = score([answer], [reference], lang="en", verbose=False)
    bert_f1 = F1.item()

    metrics.append({
        'question': question,
        'answer': answer,
        'grounding': grounding_score,
        'bert_f1': bert_f1,
        'useful': grounding_score > 0.3
    })

    print(f"{i}. Q: {question[:60]}...")
    print(f"   A: {answer}")
    print(f"   🟢 Grounding: {grounding_score:.3f} | 📈 BERT-F1: {bert_f1:.3f}\n")

# RESUMEN FINAL
df_metrics = pd.DataFrame(metrics)
print("\nTABLA RESUMEN COMPLETA:")
print(df_metrics[['question', 'answer', 'grounding', 'bert_f1', 'useful']].round(3).to_string(index=False))

print(f"\n RESUMEN FINAL:")
print(f"Grounding Promedio: {df_metrics['grounding'].mean():.3f}")
print(f"BERTScore F1 Promedio: {df_metrics['bert_f1'].mean():.3f}")
print(f"% Respuestas Útiles: {df_metrics['useful'].mean()*100:.0f}%")
print(f"Configuración: k={K_OPTIMO}, rerank={RERANK_OPTIMO}")

=== DASHBOARD MÉTRICAS COMPLETO (BERTScore) ===



Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

1. Q: How much money did Daniel Radcliffe inherit when he turned 1...
   A: £20 million ($41.1 million
   🟢 Grounding: 1.000 | 📈 BERT-F1: 0.811



Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

2. Q: How does he plan to spend his fortune?...
   A: Not found in context
   🟢 Grounding: 0.500 | 📈 BERT-F1: 0.775



...

3. Q: What does he like to buy?...
   A: things that cost about 10 pounds
   🟢 Grounding: 1.000 | 📈 BERT-F1: 0.807



Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

4. Q: Has he earned money from the Harry Potter films?...
   A: Not found in context
   🟢 Grounding: 0.500 | 📈 BERT-F1: 0.774



Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

5. Q: What other projects has he done?...
   A: He will also appear in "December Boys
   🟢 Grounding: 1.000 | 📈 BERT-F1: 0.824


TABLA RESUMEN COMPLETA:
                                                      question                                answer  grounding  bert_f1  useful
How much money did Daniel Radcliffe inherit when he turned 18?            £20 million ($41.1 million        1.0    0.811    True
                        How does he plan to spend his fortune?                  Not found in context        0.5    0.775    True
                                     What does he like to buy?      things that cost about 10 pounds        1.0    0.807    True
              Has he earned money from the Harry Potter films?                  Not found in context        0.5    0.774    True
                              What other projects has he done? He will also appear in "December Boys        1.0    0.824    True

 RESUMEN FINAL:
Grounding Promedio: 0.800
BERTScore F1 Promedio: 0.798
%